# 09m Modern image features against the paper's AlexNet (THINGS-EEG2)

**Kaggle:** GPU T4, Internet on. **Inputs:** the THINGS-EEG2 dataset (09g) and the outputs of 01 and 03b.
**Secret:** `HF_TOKEN` (for DINOv3, which is gated; without access it is skipped).

Development analysis, not a prespecified test. DINOv2, DINOv3 and SigLIP 2 give layers 4, 8 and 12 as 7 x 7 grids of
patch tokens, concatenated and passed through the paper's own pipeline (standardize, polynomial kernel, kernel PCA to
1,000 components) as AlexNet was. Then, on the 10 people: each person's own full-data encoder, borrowing, own ridge
and the personal map at k = 250 and 500. Images download from OSF (about 1.5 GB).

In [ ]:
# Which notebook this is, printed and saved with the outputs.
NOTEBOOK_ID, RUN_MODE = '09m-features', 'gpu-dev'
import json, os
print("=" * 72 + f"\n  NOTEBOOK {NOTEBOOK_ID}  |  MODE {RUN_MODE}\n" + "=" * 72)
_out = "/kaggle/working/results/metrics" if os.path.exists("/kaggle") else "results/metrics"
os.makedirs(_out, exist_ok=True)
json.dump({"notebook_id": NOTEBOOK_ID, "mode": RUN_MODE, "expected": "not checked"}, open(os.path.join(_out, f"{NOTEBOOK_ID}_run_identity.json"), "w"))

In [ ]:
import os
import time
T0 = time.monotonic()
if not os.path.exists('/kaggle'):
    raise RuntimeError('Run on Kaggle')
import importlib
import json
import os
import re
import subprocess
import sys
from pathlib import Path

PACKAGE, DIST, REPO, REF = "cross_subject_encoding", "cross-subject-encoding", "Shakkak/Borrow-then-Personalize-EEG-fMRI-Brain-Encoding-with-Limited-Labels", "v1.0.0"  # REF: the release tag a GitHub install uses
IS_KAGGLE = os.path.exists("/kaggle")


def install_args() -> list[str]:
    if not IS_KAGGLE:
        return ["-e", str(Path.cwd().parent)]  # this checkout, editable
    # a wheel of this package attached as a Kaggle Dataset, if there is one
    wheels = sorted(Path("/kaggle/input").glob(f"**/{PACKAGE}-*.whl"))
    if wheels:
        return [str(wheels[-1])]
    # otherwise install the public repository
    return [f"git+https://github.com/{REPO}.git@{REF}"]


def code_version() -> str:
    from importlib.metadata import distribution
    try:
        info = json.loads(distribution(DIST).read_text("direct_url.json") or "{}")
    except Exception:
        info = {}
    if "vcs_info" in info:
        return info["vcs_info"].get("commit_id", "")[:7]
    root = Path(info["url"].removeprefix("file://")) if "url" in info else Path.cwd().parent
    root = root.parent if root.suffix == ".whl" else root  # COMMIT sits next to the wheel
    if (root / "COMMIT").exists():
        return (root / "COMMIT").read_text().strip()
    try:
        return subprocess.run(["git", "-C", str(root), "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip() or "unknown"
    except FileNotFoundError:
        return "unknown"


try:
    pkg = importlib.import_module(PACKAGE)
except ImportError:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *install_args()], capture_output=True, text=True)
    if r.returncode:  # hide any credential in a clone URL
        raise RuntimeError("pip install failed:\n" + re.sub(r"x-access-token:[^@]+@", "x-access-token:***@", r.stderr[-3000:]))
    if not IS_KAGGLE:  # an editable install is only picked up by a fresh interpreter, so point this one at src/
        sys.path.insert(0, str(Path.cwd().parent / "src"))
    importlib.invalidate_caches()
    pkg = importlib.import_module(PACKAGE)
print(f"Kaggle: {IS_KAGGLE} | {PACKAGE} {pkg.__version__} from {Path(pkg.__file__).parent} | code version: {code_version()}")

In [ ]:
FIXED = json.loads('{"models": ["dinov2", "dinov3", "siglip2"], "layers": [4, 8, 12], "pool": 7, "batch": 64, "n_components": 1000, "people": ["sub-01", "sub-02", "sub-03", "sub-04", "sub-05", "sub-06", "sub-07", "sub-08", "sub-09", "sub-10"], "budgets": [250, 500], "draws": [13, 14, 15], "window_s": [0.1, 0.2], "map_window_ms": 100, "min_transformers": "4.56", "wall_minutes": 300}')
import subprocess
# DINOv3 needs a recent transformers; installed before transformers is first imported (a no-op if already recent)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers>={FIXED['min_transformers']}"], check=True)
import transformers
print("transformers", transformers.__version__)
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from kaggle_secrets import UserSecretsClient

from cross_subject_encoding.data import things_eeg2 as te
from cross_subject_encoding.data import things_images as ti
from cross_subject_encoding.data.splits import subset
from cross_subject_encoding.eval import eeg1_h4b as h4b
from cross_subject_encoding.eval import h4b_analysis as ha
from cross_subject_encoding.eval.paper_correlation import paper_correlation
from cross_subject_encoding.features import modern as mf
from cross_subject_encoding.models import paper_linearizing as pl
from cross_subject_encoding.models.ridge import RidgePerOutput
from cross_subject_encoding.utils import config as C
from cross_subject_encoding.utils.kaggle_export import export_to_dataset

DEVICE = "cuda"
try:
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")  # never printed
except Exception:
    TOKEN = None
roots = ti.download_images(Path("/kaggle/tmp/things_images"))
meta = te.load_image_metadata(C.EEG_RAW if te.attached(C.EEG_RAW) else None)
paths = {s: ti.image_paths(roots[s], meta, s) for s in ("training", "test")}
all_paths = paths["training"] + paths["test"]
train = np.array([True] * len(paths["training"]) + [False] * len(paths["test"]))
with np.load(C.find_artifact("paper_alexnet_kpca1000.npz")) as f:
    features = {"alexnet": (f["train"].astype(np.float64), f["test"].astype(np.float64))}
print({s: len(p) for s, p in paths.items()}, "images |", f"{(time.monotonic() - T0) / 60:.1f} min")

In [ ]:
# each modern model: grid features -> the paper's standardize / polynomial kernel / kernel PCA, fitted on training images
feature_log = {}
for name in FIXED["models"]:
    t = time.monotonic()
    torch.cuda.reset_peak_memory_stats()  # one model at a time; extract() frees each before the next loads
    try:
        x = mf.extract(all_paths, name, DEVICE, TOKEN, FIXED["batch"], tuple(FIXED["layers"]), FIXED["pool"])
    except Exception as error:  # DINOv3 without granted access, or a model that fails to load
        feature_log[name] = {"skipped": f"{type(error).__name__}: {str(error)[:300]}"}
        print(name, "SKIPPED:", feature_log[name]["skipped"])
        continue
    if not np.isfinite(x).all():
        raise ValueError(f"{name}: non-finite features")
    mean, scale = pl.standardize_stats(x, train)
    k = pl.poly_kernel(pl.gram(x, mean, scale, train, DEVICE), x.shape[1])
    z_train, z_test, _ = pl.kernel_pca(k[train], k[~train], FIXED["n_components"])
    features[name] = (z_train.astype(np.float64), z_test.astype(np.float64))
    np.savez(C.DATA_PROCESSED / f"09m_{name}_kpca1000.npz", train=z_train.astype(np.float32), test=z_test.astype(np.float32))
    feature_log[name] = {"raw_features": int(x.shape[1]), "model": mf.MODELS[name],
                         "peak_gpu_gib": torch.cuda.max_memory_allocated() / 2**30}
    del x, k
    torch.cuda.empty_cache()
    print(name, feature_log[name], "|", f"{(time.monotonic() - T0) / 60:.1f} min")
(C.METRICS_DIR / "09m_features.json").write_text(json.dumps(feature_log, indent=1))

In [ ]:
# encoders on THINGS-EEG2 for each feature set: own full-data (feature quality), others' average, own ridge, H4b
ALPHAS = [10.0 ** e for e in range(-2, 9)]
subsets = json.loads(C.find_artifact(C.EEG_K_SUBSETS).read_text())
y_train, bio = {}, {}
for s in FIXED["people"]:
    tr = te.load_part(te.subject_file(C.EEG_RAW, s, "training"))
    ts = te.load_part(te.subject_file(C.EEG_RAW, s, "test"))
    y_train[s], bio[s], times = tr["data"].mean(1), ts["data"], ts["times"]
    del tr, ts
n_ch, n_t = 17, 100
window = (times >= FIXED["window_s"][0]) & (times < FIXED["window_s"][1])
win = ha.window_index(times, FIXED["map_window_ms"])
rows = []
for fname, (z_tr, z_te) in features.items():
    t = time.monotonic()
    query = np.concatenate([z_tr, z_te])
    n_tr, n_te = len(z_tr), len(z_te)
    preds_src = {s: pl.ols_paper(z_tr, y_train[s].reshape(n_tr, -1), query) for s in FIXED["people"]}
    for s in FIXED["people"]:
        others = [q for q in FIXED["people"] if q != s]
        avg_train = np.mean([preds_src[q][:n_tr] for q in others], 0).reshape(n_tr, n_ch, n_t)
        avg_test = np.mean([preds_src[q][n_tr:] for q in others], 0).reshape(n_te, n_ch, n_t)
        preds = {"own_full|all|all": preds_src[s][n_tr:].reshape(n_te, n_ch, n_t), "avg|all|all": avg_test}
        for d in FIXED["draws"]:
            for k in FIXED["budgets"]:
                idx = np.asarray(subset(subsets, s, d, k))
                target = y_train[s][idx]
                preds[f"own_ridge|{k}|{d}"] = RidgePerOutput(ALPHAS, C.RIDGE_FOLDS, d).fit(z_tr[idx], target.reshape(k, -1)).predict(z_te).reshape(n_te, n_ch, n_t)
                preds[f"h4b|{k}|{d}"] = h4b.apply_map(h4b.fit_map(avg_train[idx], target, win, seed=d)[0], avg_test, win)
        scored = paper_correlation(preds, bio[s], n_iter=100)
        for name, curve in scored["correlation"].items():
            method, k, d = name.split("|")
            rows.append({"features": fname, "person": s, "method": method, "k": k, "draw": d, "r_100_200ms": float(curve[:, window].mean())})
    pd.DataFrame(rows).to_csv(C.METRICS_DIR / "09m_scores.csv", index=False)
    print(fname, "scored", f"{time.monotonic() - t:.0f} s |", f"{(time.monotonic() - T0) / 60:.1f} min")
frame = pd.DataFrame(rows)
m = frame.groupby(["features", "person", "method", "k"])["r_100_200ms"].mean()
def per_person(f, method, k):
    key = lambda s: (f, s, method, "all") if method in ("avg", "own_full") else (f, s, method, str(k))
    return np.array([m[key(s)] for s in FIXED["people"]])
summary = {"features": feature_log, "means": {}, "h4b_minus_avg": {}}
for f in features:
    summary["means"][f] = {"own_full": float(per_person(f, "own_full", 0).mean()), "avg": float(per_person(f, "avg", 0).mean()),
                           **{f"{meth}|{k}": float(per_person(f, meth, k).mean()) for meth in ("own_ridge", "h4b") for k in FIXED["budgets"]}}
    gain = np.mean([per_person(f, "h4b", k) - per_person(f, "avg", k) for k in FIXED["budgets"]], 0)
    summary["h4b_minus_avg"][f] = {"mean": float(gain.mean()), "positive": int((gain > 0).sum()), "n": len(gain)}
(C.METRICS_DIR / "09m_summary.json").write_text(json.dumps(summary, indent=2))
fig, ax = plt.subplots(figsize=(8, 3.8))
names = list(features)
cols = ["own_full", "avg", "own_ridge|500", "h4b|500"]
for i, c in enumerate(cols):
    ax.bar(np.arange(len(names)) + 0.2 * i, [summary["means"][f][c] for f in names], width=0.2, label=c)
ax.set(xticks=np.arange(len(names)) + 0.3, ylabel="r, 100-200 ms", title="THINGS-EEG2: feature sets (k = 500 for the few-label methods)")
ax.set_xticklabels(names)
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(C.METRICS_DIR / "09m_figure.png", dpi=150)
print(json.dumps(summary["means"], indent=1)); print(json.dumps(summary["h4b_minus_avg"], indent=1))

In [ ]:
files = [C.METRICS_DIR / n for n in ("09m_summary.json", "09m_features.json", "09m_scores.csv", "09m_figure.png")]
export_to_dataset(slug="cross-subject-brain-encoding-09m-outputs", title="XSUB 09m modern features outputs",
                  files=files, include_dirs=[C.DATA_PROCESSED], message="modern vision features through the paper pipeline")